# Colab — Build features (pipeline clásico)

Etapa más lenta del pipeline clásico (Selective Search + HOG + SIFT + BoVW sobre cada imagen train). **Corre en Colab, no en tu PC.**

Salida: `classical_features.npz` (~150 MB).

## Resume robusto

`data/processed/` se monta como **symlink a Drive**. Cada checkpoint (cada 100 imgs) se escribe **directo a Drive con write atómico** (tmp + rename). Si Colab desconecta a mitad → reabres el notebook, Run All, y reanuda desde la última imagen completada. **Nunca pierdes más de 100 imgs**, independientemente de cuándo muera Colab.

## Prerequisitos en Drive

Sube **manualmente** estos archivos a `MyDrive/grocery-detection/`:

```
MyDrive/grocery-detection/
    raw/
        d2s_images_v*.tar.xz         ← descargado de MVTec (acepta licencia)
        d2s_annotations_v*.tar.xz
    processed/
        train.json                   ← output local de scripts/prepare_splits.py
        val.json
        test.json
        codebook.pkl                 ← output local de scripts/train_codebook.py
```

`prepare_splits.py` y `train_codebook.py` son baratos (1-5 min en CPU); cómodos en local. Si tu PC tampoco aguanta eso, hay celdas opcionales al final para generarlos en Colab.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/arturmoret/GroceryTracker.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, REPO_DIR + "/scripts")
sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)

from colab_helper import (
    mount_drive, install_deps, setup_dataset,
    link_processed_to_drive, run_script,
)
print("helpers cargados")

In [ ]:
mount_drive()
install_deps()

In [ ]:
setup_dataset()  # extrae D2S a /content/d2s + symlink data/d2s

## Montar `data/processed/` directo a Drive

Tras esto, cualquier escritura en `data/processed/*` cae directo a `MyDrive/grocery-detection/processed/*`. No hay sync manual al final — los checkpoints intermedios ya están en Drive según se escriben.

In [ ]:
link_processed_to_drive()

## Build features (checkpoint cada 100 imgs, atómico, en Drive)

`--skip-svm-fit` evita entrenar la SVM aquí (eso va en `colab_train_svm.ipynb` con sample_steps=2 + n_jobs=-1).

In [ ]:
run_script("scripts/run_classical_train.py", "--skip-svm-fit")

## Listo

`classical_features.npz` ya está en `MyDrive/grocery-detection/processed/`. No hay paso de sync manual: la escritura fue directa a Drive.

Siguiente notebook: `colab_train_svm.ipynb`.

## Opcional — generar splits + codebook en Colab

Si no subiste `train.json`/`val.json`/`test.json`/`codebook.pkl` desde local, descomenta y corre estas celdas antes del `run_script` del feature build.

In [ ]:
# run_script("scripts/prepare_splits.py")

In [ ]:
# run_script("scripts/train_codebook.py")